In [ ]:
df_br['start_year'] = df_br['date_created'].dt.year
df_br['end_year'] = df_br['last_seen'].dt.year

    def get_years_in_range(row):
        return list(range(row['start_year'], row['end_year'] + 1))

    df['years_in_range'] = df.apply(get_years_in_range, axis=1)

##### Transformar dataframe conforme colunas do Monitoramento_QAr_BR

In [144]:
campos = ["UF","CIDADE","CD_MUN","ID_OEMA","ID_MMA","ID_MMA_COMPLETO","PROPRIETARIO",
          "PROP_ENTIDADE","OPERADOR","OP_ENTIDADE","FUNCIONAMENTO","CATEGORIA","METODO",
          "CALIBRACAO","MARCA","MODELO","POLUENTE","COD_POLUENTE","MOBILIDADE","REP_ESPACIAL",
          "FINALIDADE","STATUS","INICIO","FIM","LATITUDE","LONGITUDE","MONITORAR","FONTE",
          "CERTIFICACAO","COD_UF_IBGE","ANOS_MONITORADOS","BASE_DADOS","ELEVACAO"]

In [147]:
df_purple = pd.DataFrame(columns=campos) 

In [ ]:
manual_map = {
    "latitude":  "LATITUDE",
    "longitude": "LONGITUDE",
    # "name": "PROPRIETARIO",
    # "sensor_index": "ID_MMA",
    # "pm2.5": "POLUENTE",      # only if you really want that
    # add more as needed...
}

#### Importar dados do PurpleAir

##### 1. Página guia de como usar a API https://api.purpleair.com/#api-sensors-get-sensors-data
##### 2. Criar conta com e-mail Google https://develop.purpleair.com/dashboards/organization 
##### 3. Criar API key para substituir no código e ter acesso às informaçôes

In [1]:
import os, time, math, requests, pandas as pd
from datetime import datetime, timedelta, timezone

In [56]:
API_KEY = "CC7DC672-9555-11F0-BDE5-4201AC1DC121"  # put your PurpleAir Read key here
assert API_KEY and API_KEY != "YOUR_READ_KEY_HERE", "Add your PurpleAir API key"

In [58]:
BASE = "https://api.purpleair.com/v1"
HEADERS = {"X-API-Key": API_KEY}

In [81]:
# Brazil bounding box
BBOX = dict(
    nwlng = -73.990556,  # westmost
    nwlat =  5.271944,   # northmost
    selng = -34.792778,  # eastmost
    selat = -33.751944   # southmost
)

In [135]:
FIELDS = ",".join([
    "sensor_index","name","latitude","longitude", "voc","ozone1","pm1.0",
    "pm2.5","pm10.0","date_created", "last_seen","location_type", "private", "model", "altitude"
])

In [65]:
def fetch_sensors_in_bbox(fields, bbox, limit=1000, page=1):
    params = dict(fields=fields, **bbox, limit=limit, page=page)
    r = requests.get(f"{BASE}/sensors", headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

In [132]:
resp = fetch_sensors_in_bbox(FIELDS, BBOX, limit=10000, page=1)

In [136]:
cols = resp.get("fields", [])
rows = resp.get("data", [])
df = pd.DataFrame(rows, columns=cols)

In [233]:
df

,sensor_index,date_created,last_seen,private,name,location_type,model,hardware,latitude,longitude,altitude,voc,ozone1,pm1.0,pm2.5,pm10.0
0,262813,2025-01-30 23:16:47-03:00,2025-09-19 10:43:38-03:00,0,PELD-TANG,0,PA-II-SD,2.0+OPENLOG+31954 MB+DS3231+BME280+PMSX003-B+P...,-13.062653,-52.380970,1190,NaN,None,21.0,31.2,33.5
1,274372,2025-04-18 11:48:23-03:00,2025-09-19 10:43:22-03:00,0,Casa,1,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-33.359848,-70.676420,1634,97.0,None,24.2,34.4,39.1
2,275450,2025-04-24 17:36:19-03:00,2025-09-19 10:44:50-03:00,0,Vielas da Água Preta,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-23.536737,-46.692394,2435,58.0,None,19.9,26.3,29.4
3,278965,2025-05-14 16:02:09-03:00,2025-09-19 10:43:18-03:00,0,DIAMANTINO QUALIDADE DO AR,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-14.399174,-56.436085,950,87.0,None,8.4,10.1,10.9
4,280308,2025-05-22 14:09:47-03:00,2025-09-19 10:43:03-03:00,0,El Monte,0,PA-II,2.0+BME280+PMSX003-B+PMSX003-A,-33.673256,-70.986206,962,NaN,None,14.3,19.9,20.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,246457,2024-10-07 13:11:10-03:00,2025-09-19 10:43:45-03:00,0,GEOHealth_CUS_CcapacmarcaOefa3,0,PA-II-FLEX,3.0+OPENLOG+62226 MB+RV3028+BME68X+PMSX003-A+P...,-14.008719,-72.001175,11774,118.0,None,7.6,9.6,10.8
284,247259,2024-10-15 11:32:59-03:00,2025-09-18 15:20:04-03:00,0,RIO401101,0,PA-II,2.0+BME280+PMSX003-B+PMSX003-A,-22.911589,-43.756820,2,NaN,None,1.9,3.8,5.1
285,251939,2024-11-20 18:14:32-03:00,2025-09-16 10:41:36-03:00,0,prueba aranjuez,0,PA-II-FLEX,3.0+OPENLOG+7969 MB+RV3028+BME68X+PMSX003-A+PM...,-16.555443,-68.106530,10956,50.0,None,0.1,0.3,0.4
286,251937,2024-11-20 18:14:30-03:00,2025-09-19 10:43:13-03:00,0,"Universidad Privada Boliviana, Campus Achocalla",0,PA-II-FLEX,3.0+OPENLOG+31954 MB+RV3028+BME68X+PMSX003-A+P...,-16.575330,-68.126945,11548,97.0,None,9.1,13.3,14.2


##### Filtrar apenas latitudes e longitudes dentro do polígno do Brasil

In [148]:
def df_brazil_only(df, lat_col="latitude", lon_col="longitude", drop_coords=True):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326"
    )
    # alterar colunas de datas para datetime 
    df["date_created"] = pd.to_datetime(df["date_created"], unit="s", utc=True)
    df["date_created"]  = df["date_created"].dt.tz_convert("America/Sao_Paulo")
    df["last_seen"] = pd.to_datetime(df["last_seen"], unit="s", utc=True)
    df["last_seen"]  = df["last_seen"].dt.tz_convert("America/Sao_Paulo")
    
    # polígono do país para extrair contornos
    url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    world = gpd.read_file(url)
    brazil_poly = world.loc[world["SOVEREIGNT"] == "Brazil", "geometry"].iloc[0]

    # cria mask para manter somente lat e long dentro do contorno
    mask = gdf.within(brazil_poly)
    gdf = gdf[mask].copy()
    
    return gdf

In [237]:
df_br = df_brazil_only(df)   # df_br contém somente as linhas lat e long dentro do Brasil
print("Shape sensores PurpleAir no Brasil: " +str(df_br.shape))
df_br.head()

Shape sensores PurpleAir no Brasil: (176, 17)


,sensor_index,date_created,last_seen,private,name,location_type,model,hardware,latitude,longitude,altitude,voc,ozone1,pm1.0,pm2.5,pm10.0,geometry
0,262813,2025-01-30 23:16:47-03:00,2025-09-19 10:43:38-03:00,0,PELD-TANG,0,PA-II-SD,2.0+OPENLOG+31954 MB+DS3231+BME280+PMSX003-B+P...,-13.062653,-52.380970,1190,NaN,None,21.0,31.2,33.5,POINT (-52.38097 -13.06265)
2,275450,2025-04-24 17:36:19-03:00,2025-09-19 10:44:50-03:00,0,Vielas da Água Preta,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-23.536737,-46.692394,2435,58.0,None,19.9,26.3,29.4,POINT (-46.69239 -23.53674)
3,278965,2025-05-14 16:02:09-03:00,2025-09-19 10:43:18-03:00,0,DIAMANTINO QUALIDADE DO AR,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-14.399174,-56.436085,950,87.0,None,8.4,10.1,10.9,POINT (-56.43608 -14.39917)
21,25541,2019-01-22 22:03:48-02:00,2025-09-19 10:43:48-03:00,0,MPAC_PTA_01_Sec.infraestrutura,0,PA-II-SD,2.0+OPENLOG+NO-DISK+BME280+PMSX003-B+PMSX003-A,-9.727080,-67.698020,677,NaN,None,4.7,6.5,6.9,POINT (-67.69802 -9.72708)
22,25551,2019-01-22 22:04:35-02:00,2025-09-19 10:43:44-03:00,0,MPAC_FIJ_01_promotoria,0,PA-II-SD,2.0+OPENLOG+NO-DISK+DS3231+BME280+PMSX003-B+PM...,-8.170079,-70.355030,525,NaN,None,2.9,4.1,4.2,POINT (-70.35503 -8.17008)


In [152]:
df_br['start_year'] = df_br['date_created'].dt.year
df_br['end_year'] = df_br['last_seen'].dt.year

def get_years_in_range(row): # extrair todos os anos de medição de cada sensor
    return list(range(row['start_year'], row['end_year'] + 1))

df_br['years_in_range'] = df_br.apply(get_years_in_range, axis=1)
df_br.head()

,sensor_index,date_created,last_seen,private,name,location_type,model,hardware,latitude,longitude,altitude,voc,ozone1,pm1.0,pm2.5,pm10.0,geometry,start_year,end_year,years_in_range
0,262813,2025-01-30 23:16:47-03:00,2025-09-19 10:43:38-03:00,0,PELD-TANG,0,PA-II-SD,2.0+OPENLOG+31954 MB+DS3231+BME280+PMSX003-B+P...,-13.062653,-52.380970,1190,NaN,None,21.0,31.2,33.5,POINT (-52.38097 -13.06265),2025,2025,[2025]
2,275450,2025-04-24 17:36:19-03:00,2025-09-19 10:44:50-03:00,0,Vielas da Água Preta,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-23.536737,-46.692394,2435,58.0,None,19.9,26.3,29.4,POINT (-46.69239 -23.53674),2025,2025,[2025]
3,278965,2025-05-14 16:02:09-03:00,2025-09-19 10:43:18-03:00,0,DIAMANTINO QUALIDADE DO AR,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-14.399174,-56.436085,950,87.0,None,8.4,10.1,10.9,POINT (-56.43608 -14.39917),2025,2025,[2025]
21,25541,2019-01-22 22:03:48-02:00,2025-09-19 10:43:48-03:00,0,MPAC_PTA_01_Sec.infraestrutura,0,PA-II-SD,2.0+OPENLOG+NO-DISK+BME280+PMSX003-B+PMSX003-A,-9.727080,-67.698020,677,NaN,None,4.7,6.5,6.9,POINT (-67.69802 -9.72708),2019,2025,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]"
22,25551,2019-01-22 22:04:35-02:00,2025-09-19 10:43:44-03:00,0,MPAC_FIJ_01_promotoria,0,PA-II-SD,2.0+OPENLOG+NO-DISK+DS3231+BME280+PMSX003-B+PM...,-8.170079,-70.355030,525,NaN,None,2.9,4.1,4.2,POINT (-70.35503 -8.17008),2019,2025,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]"


In [155]:
import numpy as np

In [158]:
cols = ["voc", "ozone1", "pm1.0", "pm2.5", "pm10.0"]  
labels = {"voc":"VOC", "pm1.0":"PM1", "pm2.5":"PM25", "pm10.0":"PM10", "ozone1":"OZONE"}

def get_pollutants(df_br, cols, labels):
    df_br["pollutants_all"] = df_br[cols].apply(
    lambda r: ",".join(labels[c] for c in cols if r[c] is not None) or pd.NA,
    axis=1
    )

    return df_br

In [159]:
df_br = get_pollutants(df_br, cols, labels)
df_br.head()

,sensor_index,date_created,last_seen,private,name,location_type,model,hardware,latitude,longitude,...,ozone1,pm1.0,pm2.5,pm10.0,geometry,start_year,end_year,years_in_range,pollutant,pollutants_all
0,262813,2025-01-30 23:16:47-03:00,2025-09-19 10:43:38-03:00,0,PELD-TANG,0,PA-II-SD,2.0+OPENLOG+31954 MB+DS3231+BME280+PMSX003-B+P...,-13.062653,-52.380970,...,None,21.0,31.2,33.5,POINT (-52.38097 -13.06265),2025,2025,[2025],PM1,"VOC,PM1,PM25,PM10"
2,275450,2025-04-24 17:36:19-03:00,2025-09-19 10:44:50-03:00,0,Vielas da Água Preta,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-23.536737,-46.692394,...,None,19.9,26.3,29.4,POINT (-46.69239 -23.53674),2025,2025,[2025],VOC,"VOC,PM1,PM25,PM10"
3,278965,2025-05-14 16:02:09-03:00,2025-09-19 10:43:18-03:00,0,DIAMANTINO QUALIDADE DO AR,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-14.399174,-56.436085,...,None,8.4,10.1,10.9,POINT (-56.43608 -14.39917),2025,2025,[2025],VOC,"VOC,PM1,PM25,PM10"
21,25541,2019-01-22 22:03:48-02:00,2025-09-19 10:43:48-03:00,0,MPAC_PTA_01_Sec.infraestrutura,0,PA-II-SD,2.0+OPENLOG+NO-DISK+BME280+PMSX003-B+PMSX003-A,-9.727080,-67.698020,...,None,4.7,6.5,6.9,POINT (-67.69802 -9.72708),2019,2025,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",PM1,"VOC,PM1,PM25,PM10"
22,25551,2019-01-22 22:04:35-02:00,2025-09-19 10:43:44-03:00,0,MPAC_FIJ_01_promotoria,0,PA-II-SD,2.0+OPENLOG+NO-DISK+DS3231+BME280+PMSX003-B+PM...,-8.170079,-70.355030,...,None,2.9,4.1,4.2,POINT (-70.35503 -8.17008),2019,2025,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",PM1,"VOC,PM1,PM25,PM10"


##### Extrair nome da cidade e estado baseado na latitude e longitude

In [173]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
from geopy.extra.rate_limiter import RateLimiter

In [174]:
geolocator = Nominatim(user_agent="bixtech-airquality", timeout=10)
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1.1, max_retries=2, error_wait_seconds=2)

In [175]:
def get_city_name(latitude, longitude):
    try:
        location = geolocator.reverse((latitude, longitude), exactly_one=True)
        if location and location.address:
            address_parts = location.raw.get('address', {})
            return address_parts.get('city')
        return None
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        print(f"Error geocoding {latitude}, {longitude}: {e}")
        return None

In [176]:
def get_uf_name(latitude, longitude):
    try:
        location = geolocator.reverse((latitude, longitude), exactly_one=True)
        if location and location.address:
            address_parts = location.raw.get('address', {})
            return address_parts.get('state')
        return None
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        print(f"Error geocoding {latitude}, {longitude}: {e}")
        return None

In [177]:
df_br['CIDADE'] = df_br.apply(lambda row: get_city_name(row['latitude'], row['longitude']), axis=1)
df_br['UF'] = df_br.apply(lambda row: get_uf_name(row['latitude'], row['longitude']), axis=1)

,sensor_index,date_created,last_seen,private,name,location_type,model,hardware,latitude,longitude,...,pm2.5,pm10.0,geometry,start_year,end_year,years_in_range,pollutant,pollutants_all,CIDADE,UF
0,262813,2025-01-30 23:16:47-03:00,2025-09-19 10:43:38-03:00,0,PELD-TANG,0,PA-II-SD,2.0+OPENLOG+31954 MB+DS3231+BME280+PMSX003-B+P...,-13.062653,-52.380970,...,31.2,33.5,POINT (-52.38097 -13.06265),2025,2025,[2025],PM1,"VOC,PM1,PM25,PM10",None,Mato Grosso
2,275450,2025-04-24 17:36:19-03:00,2025-09-19 10:44:50-03:00,0,Vielas da Água Preta,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-23.536737,-46.692394,...,26.3,29.4,POINT (-46.69239 -23.53674),2025,2025,[2025],VOC,"VOC,PM1,PM25,PM10",São Paulo,São Paulo
3,278965,2025-05-14 16:02:09-03:00,2025-09-19 10:43:18-03:00,0,DIAMANTINO QUALIDADE DO AR,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-14.399174,-56.436085,...,10.1,10.9,POINT (-56.43608 -14.39917),2025,2025,[2025],VOC,"VOC,PM1,PM25,PM10",None,Mato Grosso
21,25541,2019-01-22 22:03:48-02:00,2025-09-19 10:43:48-03:00,0,MPAC_PTA_01_Sec.infraestrutura,0,PA-II-SD,2.0+OPENLOG+NO-DISK+BME280+PMSX003-B+PMSX003-A,-9.727080,-67.698020,...,6.5,6.9,POINT (-67.69802 -9.72708),2019,2025,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",PM1,"VOC,PM1,PM25,PM10",None,Acre
22,25551,2019-01-22 22:04:35-02:00,2025-09-19 10:43:44-03:00,0,MPAC_FIJ_01_promotoria,0,PA-II-SD,2.0+OPENLOG+NO-DISK+DS3231+BME280+PMSX003-B+PM...,-8.170079,-70.355030,...,4.1,4.2,POINT (-70.35503 -8.17008),2019,2025,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",PM1,"VOC,PM1,PM25,PM10",None,Acre


In [179]:
df_br.head()

,sensor_index,date_created,last_seen,private,name,location_type,model,hardware,latitude,longitude,...,pm2.5,pm10.0,geometry,start_year,end_year,years_in_range,pollutant,pollutants_all,CIDADE,UF
0,262813,2025-01-30 23:16:47-03:00,2025-09-19 10:43:38-03:00,0,PELD-TANG,0,PA-II-SD,2.0+OPENLOG+31954 MB+DS3231+BME280+PMSX003-B+P...,-13.062653,-52.380970,...,31.2,33.5,POINT (-52.38097 -13.06265),2025,2025,[2025],PM1,"VOC,PM1,PM25,PM10",None,Mato Grosso
2,275450,2025-04-24 17:36:19-03:00,2025-09-19 10:44:50-03:00,0,Vielas da Água Preta,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-23.536737,-46.692394,...,26.3,29.4,POINT (-46.69239 -23.53674),2025,2025,[2025],VOC,"VOC,PM1,PM25,PM10",São Paulo,São Paulo
3,278965,2025-05-14 16:02:09-03:00,2025-09-19 10:43:18-03:00,0,DIAMANTINO QUALIDADE DO AR,0,PA-II-ZEN,3.0+OPENLOG+NO-DISK+RV3028+BME68X+KX122+PMSX00...,-14.399174,-56.436085,...,10.1,10.9,POINT (-56.43608 -14.39917),2025,2025,[2025],VOC,"VOC,PM1,PM25,PM10",None,Mato Grosso
21,25541,2019-01-22 22:03:48-02:00,2025-09-19 10:43:48-03:00,0,MPAC_PTA_01_Sec.infraestrutura,0,PA-II-SD,2.0+OPENLOG+NO-DISK+BME280+PMSX003-B+PMSX003-A,-9.727080,-67.698020,...,6.5,6.9,POINT (-67.69802 -9.72708),2019,2025,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",PM1,"VOC,PM1,PM25,PM10",None,Acre
22,25551,2019-01-22 22:04:35-02:00,2025-09-19 10:43:44-03:00,0,MPAC_FIJ_01_promotoria,0,PA-II-SD,2.0+OPENLOG+NO-DISK+DS3231+BME280+PMSX003-B+PM...,-8.170079,-70.355030,...,4.1,4.2,POINT (-70.35503 -8.17008),2019,2025,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",PM1,"VOC,PM1,PM25,PM10",None,Acre


In [69]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

In [141]:
import folium

m = folium.Map(location=[-14.235, -51.925], zoom_start=4)

for lat, lon in zip(df_br["latitude"], df_br["longitude"]):
    if pd.notna(lat) and pd.notna(lon):
        folium.CircleMarker(location=[lat, lon], radius=3).add_to(m)

In [232]:
m

##### Transformar dataframe conforme colunas do Monitoramento_QAr_BR

In [185]:
campos = ["UF","CIDADE","CD_MUN","ID_OEMA","ID_MMA","ID_MMA_COMPLETO","PROPRIETARIO",
          "PROP_ENTIDADE","OPERADOR","OP_ENTIDADE","FUNCIONAMENTO","CATEGORIA","METODO",
          "CALIBRACAO","MARCA","MODELO","POLUENTE","COD_POLUENTE","MOBILIDADE","REP_ESPACIAL",
          "FINALIDADE","STATUS","INICIO","FIM","LATITUDE","LONGITUDE","MONITORAR","FONTE",
          "CERTIFICACAO","COD_UF_IBGE","ANOS_MONITORADOS","BASE_DADOS","ELEVACAO"]

In [187]:
df_purple = pd.DataFrame(columns=campos) 

In [188]:
manual_map = {
    "latitude":  "LATITUDE",
    "longitude": "LONGITUDE",
    "name": "ID_OEMA",
    "private": "PROP_ENTIDADE",
    "model": "MODELO",
    "years_in_range": "ANOS_MONITORADOS",
    "pollutants_all": "POLUENTE",
    "date_created": "INICIO",
    "last_seen": "FIM",
    "altitude": "ELEVACAO"
}

df_br = df_br.rename(columns=manual_map)

In [196]:
df_purple = df_purple.reindex(index=df_br.index)  
common = [c for c in campos if c in df_br.columns]
df_purple.loc[:, common] = df_br[common].values
df_purple.head()

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,FIM,LATITUDE,LONGITUDE,MONITORAR,FONTE,CERTIFICACAO,COD_UF_IBGE,ANOS_MONITORADOS,BASE_DADOS,ELEVACAO
0,Mato Grosso,None,NaN,PELD-TANG,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:43:38-03:00,-13.062653,-52.38097,NaN,PurpleAir 2025,NaN,NaN,[2025],NaN,1190
2,São Paulo,São Paulo,NaN,Vielas da Água Preta,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:44:50-03:00,-23.536737,-46.692394,NaN,PurpleAir 2025,NaN,NaN,[2025],NaN,2435
3,Mato Grosso,None,NaN,DIAMANTINO QUALIDADE DO AR,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:43:18-03:00,-14.399174,-56.436085,NaN,PurpleAir 2025,NaN,NaN,[2025],NaN,950
21,Acre,None,NaN,MPAC_PTA_01_Sec.infraestrutura,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:43:48-03:00,-9.72708,-67.69802,NaN,PurpleAir 2025,NaN,NaN,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",NaN,677
22,Acre,None,NaN,MPAC_FIJ_01_promotoria,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:43:44-03:00,-8.170079,-70.35503,NaN,PurpleAir 2025,NaN,NaN,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",NaN,525


In [192]:
defaults = {
    "FUNCIONAMENTO": "Automatica",
    "CATEGORIA":     "Indicativa",
    "MARCA":         "PurpleAir",
    "FONTE":         "PurpleAir 2025",
}

df_purple = df_purple.fillna(value=defaults)

In [193]:
map_prop = {
    0: "Publico", 1: "Privada",
    False: "Publico", True: "Privado",
    "0": "Publico", "1": "Privado",
}

df_purple["PROP_ENTIDADE"] = df_purple["PROP_ENTIDADE"].replace(map_prop)
df_purple["PROP_ENTIDADE"] = df_purple["PROP_ENTIDADE"].astype("string")

In [204]:
def years_to_list(v):
    if isinstance(v, (list, tuple, set)):
        return ",".join(str(x) for x in v)              
    return v                                            

df_purple["ANOS_MONITORADOS"] = df_purple["ANOS_MONITORADOS"].apply(years_to_list)

In [206]:
import unicodedata

In [207]:
def _key(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    s = unicodedata.normalize("NFKD", s).encode("ASCII","ignore").decode("ASCII")
    return s.lower()

In [208]:
# mapa: nome da UF -> sigla
_name_to_sigla = {
    "acre":"AC","alagoas":"AL","amapa":"AP","amazonas":"AM","bahia":"BA","ceara":"CE",
    "distrito federal":"DF","espirito santo":"ES","goias":"GO","maranhao":"MA",
    "mato grosso":"MT","mato grosso do sul":"MS","minas gerais":"MG","para":"PA",
    "paraiba":"PB","parana":"PR","pernambuco":"PE","piaui":"PI","rio de janeiro":"RJ",
    "rio grande do norte":"RN","rio grande do sul":"RS","rondonia":"RO","roraima":"RR",
    "santa catarina":"SC","sao paulo":"SP","sergipe":"SE","tocantins":"TO"
}

# também aceitar quando a coluna já vier com a sigla
for sig in list(_name_to_sigla.values()):
    _name_to_sigla[sig.lower()] = sig

# códigos IBGE por sigla
_sigla_to_ibge = {
    "AC":"12","AL":"27","AP":"16","AM":"13","BA":"29","CE":"23","DF":"53","ES":"32",
    "GO":"52","MA":"21","MT":"51","MS":"50","MG":"31","PA":"15","PB":"25","PR":"41",
    "PE":"26","PI":"22","RN":"24","RS":"43","RJ":"33","RO":"11","RR":"14","SC":"42",
    "SP":"35","SE":"28","TO":"17"
}

In [209]:
# substituir nome por sigla na coluna UF
df_purple["UF"] = df_purple["UF"].apply(lambda x: _name_to_sigla.get(_key(x), pd.NA))

# preencher COD_UF_IBGE a partir da sigla
df_purple["COD_UF_IBGE"] = df_purple["UF"].map(_sigla_to_ibge)

# dtypes opcionais
df_purple["UF"] = df_purple["UF"].astype("string")
df_purple["COD_UF_IBGE"] = df_purple["COD_UF_IBGE"].astype("string")

In [210]:
df_purple.head()

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,FIM,LATITUDE,LONGITUDE,MONITORAR,FONTE,CERTIFICACAO,COD_UF_IBGE,ANOS_MONITORADOS,BASE_DADOS,ELEVACAO
0,MT,None,NaN,PELD-TANG,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:43:38-03:00,-13.062653,-52.38097,NaN,PurpleAir 2025,NaN,51,2025,NaN,1190
2,SP,São Paulo,NaN,Vielas da Água Preta,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:44:50-03:00,-23.536737,-46.692394,NaN,PurpleAir 2025,NaN,35,2025,NaN,2435
3,MT,None,NaN,DIAMANTINO QUALIDADE DO AR,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:43:18-03:00,-14.399174,-56.436085,NaN,PurpleAir 2025,NaN,51,2025,NaN,950
21,AC,None,NaN,MPAC_PTA_01_Sec.infraestrutura,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:43:48-03:00,-9.72708,-67.69802,NaN,PurpleAir 2025,NaN,12,"2019,2020,2021,2022,2023,2024,2025",NaN,677
22,AC,None,NaN,MPAC_FIJ_01_promotoria,NaN,NaN,NaN,0,NaN,NaN,...,2025-09-19 10:43:44-03:00,-8.170079,-70.35503,NaN,PurpleAir 2025,NaN,12,"2019,2020,2021,2022,2023,2024,2025",NaN,525


In [245]:
df_purple.groupby(["UF"]).count()

,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,FUNCIONAMENTO,...,FIM,LATITUDE,LONGITUDE,MONITORAR,FONTE,CERTIFICACAO,COD_UF_IBGE,ANOS_MONITORADOS,BASE_DADOS,ELEVACAO
UF,,,,,,,,,,,,,,,,,,,,,
AC,5,0,15,0,0,0,15,0,0,15,...,15,15,15,0,15,0,15,15,0,15
AM,19,0,65,0,0,0,65,0,0,65,...,65,65,65,0,65,0,65,65,0,65
AP,2,0,6,0,0,0,6,0,0,6,...,6,6,6,0,6,0,6,6,0,6
BA,1,0,1,0,0,0,1,0,0,1,...,1,1,1,0,1,0,1,1,0,1
CE,1,0,1,0,0,0,1,0,0,1,...,1,1,1,0,1,0,1,1,0,1
DF,3,0,5,0,0,0,5,0,0,5,...,5,5,5,0,5,0,5,5,0,5
MS,1,0,2,0,0,0,2,0,0,2,...,2,2,2,0,2,0,2,2,0,2
MT,6,0,19,0,0,0,19,0,0,19,...,19,19,19,0,19,0,19,19,0,19
PA,18,0,23,0,0,0,23,0,0,23,...,23,23,23,0,23,0,23,23,0,23


In [211]:
def explode_pollutants(df_purple, col="POLUENTE"):
    out = df_purple.copy()

    # turn "VOC,PM1, PM25 ,PM10" into ["VOC","PM1","PM25","PM10"]
    out[col] = (
        out[col]
        .astype("string")
        .fillna("")
        .apply(lambda s: [p.strip() for p in s.split(",") if p.strip()])
    )

    # explode to one row per pollutant
    out = out.explode(col, ignore_index=True)

    return out.reset_index(drop=True)

df_purple_exp = explode_pollutants(df_purple)

In [215]:
print("CWD:", Path.cwd())

CWD: /home/nobre/Notebooks/RQAR_2025_book/scripts


In [216]:
from pathlib import Path

base = Path.cwd().parent  # .../RQAR_2025_book
out_dir = base / "data" / "DADOS_ESTACOES"
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "PurpleAirStations.csv"
df_purple_exp.to_csv(out_file, index=False, encoding="utf-8")
print("Saved to:", out_file.resolve())

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/PurpleAirStations.csv


#### Comparar dados das UFs com base de dados PurpleAir

In [227]:
base = Path.cwd().parent  
out_dir = base / "data" 
out_dir.mkdir(parents=True, exist_ok=True)

df_mma = pd.read_csv(out_dir / "Monitoramento_QAr_BR.csv")
df_mma.columns

Index(['UF', 'CIDADE', 'CD_MUN', 'ID_OEMA', 'ID_MMA', 'ID_MMA_COMPLETO',
       'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE',
       'FUNCIONAMENTO', 'CATEGORIA', 'METODO', 'CALIBRACAO', 'MARCA', 'MODELO',
       'POLUENTE', 'COD_POLUENTE', 'MOBILIDADE', 'REP_ESPACIAL', 'FINALIDADE',
       'STATUS', 'INICIO', 'FIM', 'LATITUDE', 'LONGITUDE', 'MONITORAR',
       'FONTE', 'CERTIFICACAO', 'COD_UF_IBGE'],
      dtype='object')

In [247]:
id_col = "ID_OEMA"

def norm(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.upper()

# normalized ID columns
mma_id = norm(df_mma[id_col])
pur_id = norm(df_purple_exp[id_col])

# IDs present in both dataframes
common_ids = pd.Index(mma_id.dropna()).intersection(pur_id.dropna())
print("IDs present in both:", len(common_ids))

# rows involved on each side
mma_overlap = df_mma[mma_id.isin(common_ids)]
pur_overlap = df_purple_exp[pur_id.isin(common_ids)]
print("Rows in df_mma with overlapping ID_OEMA:", len(mma_overlap))
print("Rows in df_purple_exp with overlapping ID_OEMA:", len(pur_overlap))

# summary count per ID showing how many times it appears in each df
both = pd.concat(
    [
        pd.DataFrame({id_col: mma_id, "_src": "mma"}),
        pd.DataFrame({id_col: pur_id, "_src": "purple"}),
    ],
    ignore_index=True,
)
summary = (
    both.dropna(subset=[id_col])
        .groupby([id_col, "_src"]).size()
        .unstack(fill_value=0)
        .query("mma > 0 and purple > 0")
        .sort_index()
)
print("Overlapping unique IDs:", summary.shape[0])

IDs present in both: 44
Rows in df_mma with overlapping ID_OEMA: 71
Rows in df_purple_exp with overlapping ID_OEMA: 180
Overlapping unique IDs: 44


##### Overlap de mapas - Coordenadas MMA e PurpleAir

In [249]:
def add_points(df, lat_col="latitude", lon_col="longitude", color="blue", name="Layer"):
    fg = folium.FeatureGroup(name=name, show=True)
    for lat, lon in zip(df[lat_col], df[lon_col]):
        if pd.notna(lat) and pd.notna(lon):
            folium.CircleMarker(
                location=[lat, lon],
                radius=3,
                color=color,
                fill=True,
                fill_opacity=0.8,
                opacity=0.8
            ).add_to(fg)
    fg.add_to(m)

# Plot your two dataframes
add_points(df_purple,  lat_col="LATITUDE", lon_col="LONGITUDE", color="blue",  name="PurpleAir")
add_points(df_mma, lat_col="LATITUDE", lon_col="LONGITUDE", color="red",   name="MMA")

# Layer control to toggle visibility
folium.LayerControl(collapsed=False).add_to(m)

m  

In [250]:
from shapely.geometry import Point
import folium

# 1) Build GeoDataFrames (WGS84)
gdf_br  = gpd.GeoDataFrame(
    df_br.copy(),
    geometry=gpd.points_from_xy(df_purple["LONGITUDE"], df_purple["LATITUDE"]),
    crs="EPSG:4326"
)
gdf_br["source"] = "br"

gdf_mma = gpd.GeoDataFrame(
    df_mma.copy(),
    geometry=gpd.points_from_xy(df_mma["LONGITUDE"], df_mma["LATITUDE"]),
    crs="EPSG:4326"
)
gdf_mma["source"] = "mma"

# Combine
gdf = pd.concat([gdf_br, gdf_mma], ignore_index=True)

# 2) Project to a meter-based CRS covering Brazil (SIRGAS 2000 / Brazil Polyconic)
gdf_m = gdf.to_crs("EPSG:5880")

# 3) Build 100 m buffers around each point
buffers = gdf_m.copy()
buffers["geometry"] = buffers.geometry.buffer(100)

# 4) Spatial join to find points that fall within any buffer
#    This includes the point’s own buffer, so groups with size > 1 mean overlap with another point.
join = gpd.sjoin(gdf_m, buffers[["geometry"]], predicate="within", how="left", lsuffix="pt", rsuffix="buf")

# Count how many points fall inside each buffer (by the buffer owner’s index)
counts = join.groupby("index_right").size()

# Mark overlapping points: any buffer that contains 2 or more points
overlapping_idx = counts[counts > 1].index  # these are buffer-owner indices that overlap
gdf_m["overlap"] = gdf_m.index.isin(overlapping_idx)

# 5) Back to lat/lon for Folium plotting and listing
gdf_plot = gdf_m.to_crs("EPSG:4326")

# 6) Build Folium map
m = folium.Map(location=[-14.235, -51.925], zoom_start=4, tiles="CartoDB positron")

for _, row in gdf_plot.iterrows():
    lat, lon = row.geometry.y, row.geometry.x
    if pd.notna(lat) and pd.notna(lon):
        color = "yellow" if row["overlap"] else ("blue" if row["source"] == "br" else "red")
        folium.CircleMarker(
            location=[lat, lon],
            radius=4,
            color=color,
            fill=True,
            fill_opacity=0.9,
            opacity=0.9,
            popup=f"{row['source']} | {lat:.6f}, {lon:.6f}"
        ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# 7) Print lat/lon of overlapping points
overlap_pts = gdf_plot[gdf_plot["overlap"]][["source"]].copy()
overlap_pts["latitude"] = gdf_plot[gdf_plot["overlap"]].geometry.y
overlap_pts["longitude"] = gdf_plot[gdf_plot["overlap"]].geometry.x

print("Overlapping points (within 100 m of at least one other point):")
print(overlap_pts.to_string(index=False))

m  # display in a notebook


KeyError: 'index_right'

#### Importar série histórica de dados

In [11]:
from dateutil import tz

In [129]:
def month_ranges_2024():
    return [(datetime(2024,m,1,tzinfo=timezone.utc),
             datetime(2024,m+1,1,tzinfo=timezone.utc) if m<12 else datetime(2025,1,1,tzinfo=timezone.utc))
            for m in range(1,13)]

history_fields = "pm2.5_atm"  
AVERAGE_MIN = 60

hist_frames = []
for sid in df_br["sensor_index"].head(20):  # raise gradually
    for start, end in month_ranges_2024():
        try:
            h = fetch_sensor_history(int(sid), history_fields, start, end, average=AVERAGE_MIN)
            rows, cols = h.get("data", []), h.get("fields", [])
            if not rows:
                continue
            hdf = pd.DataFrame(rows, columns=cols)
            hdf["sensor_index"] = int(sid)
            hist_frames.append(hdf)
            time.sleep(0.4)
        except requests.HTTPError as e:
            print(f"Sensor {sid} {start:%Y-%m} skipped:", e.response.text)
            time.sleep(0.6)

hist = pd.concat(hist_frames, ignore_index=True) if hist_frames else pd.DataFrame()
print(hist.shape)

(113334, 3)


In [12]:
def fetch_sensor_history(sensor_index, fields, start, end, average=10):
    params = {
        "fields": fields,
        "start_timestamp": int(start.timestamp()),
        "end_timestamp": int(end.timestamp()),
        "average": average
    }
    url = f"{BASE}/sensors/{sensor_index}/history"
    r = requests.get(url, headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

In [14]:
import pandas as pd

In [26]:
history_fields = "pm2.5_atm"  # no time_stamp here
AVERAGE_MIN = 60  # hourly averages

def month_ranges_2024():
    return [(datetime(2024, m, 1, tzinfo=timezone.utc),
             datetime(2024, m+1, 1, tzinfo=timezone.utc) if m < 12 else datetime(2025, 1, 1, tzinfo=timezone.utc))
            for m in range(1, 13)]

hist_frames = []
for sid in df["sensor_index"].head(3):
    for start, end in month_ranges_2024():
        try:
            h = fetch_sensor_history(int(sid), history_fields, start, end, average=AVERAGE_MIN)
            cols = h.get("fields", [])
            rows = h.get("data", [])
            if not rows:
                continue
            hdf = pd.DataFrame(rows, columns=cols)
            # The response already includes a timestamp column (often named time_stamp)
            hdf["sensor_index"] = int(sid)  # add ID yourself
            hist_frames.append(hdf)
            time.sleep(0.4)
        except requests.HTTPError as e:
            print(f"Sensor {sid} {start:%Y-%m} skipped:", getattr(e, "response", None).text if getattr(e, "response", None) else e)
            time.sleep(0.6)

In [27]:
hist = pd.concat(hist_frames, ignore_index=True) if hist_frames else pd.DataFrame()
print(hist)

Empty DataFrame
Columns: []
Index: []
